# Q1: Were planned stops actually serviced? (Missed Stops)
Type A — Logged but not completed

This notebook computes the Q1 analysis for missed stops using the actual available tables.

### Tables used
| Table | Role |
|---|---|
| `trip_stops_long.csv` | Core stop-level records: `completed`, `completed_diff`, etc. |
| `trips.csv` | Trip metadata: `status`, `riders_count` |
| `route_stops_clean.csv` | Stop dimension: `stop_types`, `is_anchor`, `is_pickup`, `is_dropoff` |

### Output files
- **`stop_level_diagnostic.csv`** — one row per stop

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

---
## Phase 1: Define Analysis Baseline (Filtering & Baseline)

**Goal:** Build a clean, analysis-ready baseline of real service records.

**Key column mapping**
- `status` → in `trips.csv`
- `riders_count` → in `trips.csv`
- `stop_types` → sourced from `route_stops_clean.csv`

**Filters applied:**
1. `status == 'completed'` — exclude pending / cancelled trips.

In [ ]:
# --- Load tables ---
PATH = "<PROCESSED_DATA_DIR>"

trip_stops   = pd.read_csv(PATH + '/trip_stops_long.csv')
trips        = pd.read_csv(PATH + '/trips.csv')
route_stops  = pd.read_csv(PATH + '/route_stops_clean.csv')
trip_master  = pd.read_csv(PATH + '/trip_master_join.csv')

print('trip_stops_long  :', trip_stops.shape)
print('trips            :', trips.shape)
print('route_stops_clean:', route_stops.shape)
print('trip_master_join :', trip_master.shape)

In [ ]:
# --- Merge trip_stops with trips on trip_id ---
df = trip_stops.merge(
    trip_master[['trip_id', 'status', 'riders_count', 'route_id']],
    on='trip_id', how='left'
)

print('Rows in total:', len(df))

---
## Phase 2: Stop Service Reliability Score (SSRS — Stop Level)

**Goal:** Estimate the miss probability for each `route_stop_id`.

**Formula:**
> `SSRS = (completed == True count / total appearances) × 100`

- `stop_types`, `is_anchor`, `is_pickup`, `is_dropoff`, `is_school`, `final_anchor`
  are all available in `route_stops_clean.csv` and be attached for richer diagnostics.
- `rsi_stop_count`, `rsi_completed_count`, `rsi_incomplete_count`, `rsi_avg_completed_diff`
  are pre-aggregated in `trip_stops_long.csv` and serve as a cross-check on SSRS.

In [ ]:
# --- Compute SSRS per route_stop_id ---
ssrs = (
    df.groupby('route_stop_id')['completed']
    .agg(
        total_dispatches='count',
        completed_count='sum'
    )
    .reset_index()
)

ssrs['ssrs'] = (ssrs['completed_count'] / ssrs['total_dispatches']) * 100

# Type A count: stops that were dispatched but NOT completed (missed)
ssrs['type_a_count'] = ssrs['total_dispatches'] - ssrs['completed_count']

print('SSRS computed for', len(ssrs), 'unique stops')
ssrs.sort_values('ssrs').head(10)

In [ ]:
# --- Output 2: stop_level_diagnostic.csv ---

# Average completed_diff for completed stops only
avg_diff = (
    df[df['completed'] == True]
    .groupby('route_stop_id')['completed_diff']
    .mean()
    .reset_index()
    .rename(columns={'completed_diff': 'avg_completed_diff'})
)

# Merge SSRS + avg timing + stop metadata from route_stops_clean
# route_stops_clean.csv has: stop_types, is_anchor, is_pickup, is_dropoff,
#                             is_school, is_anchor_stop, final_anchor
stop_meta_cols = [
    'route_stop_id', 'stop_types', 'is_anchor', 'is_pickup',
    'is_dropoff', 'is_school', 'is_anchor_stop', 'final_anchor'
]
stop_meta_cols = [c for c in stop_meta_cols if c in route_stops.columns]

stop_report = (
    ssrs
    .merge(avg_diff, on='route_stop_id', how='left')
    .merge(route_stops[stop_meta_cols], on='route_stop_id', how='left')
)

# Select and order final columns
stop_output_cols = [
    'route_stop_id', 'stop_types',
    'is_anchor', 'is_pickup', 'is_dropoff', 'is_school', 'is_anchor_stop', 'final_anchor',
    'ssrs', 'avg_completed_diff',
    'total_dispatches', 'completed_count', 'type_a_count',
]
# Append RSI cross-check columns if present
rsi_check_cols = ['rsi_stop_count', 'rsi_completed_count', 'rsi_incomplete_count',
                  'rsi_avg_completed_diff', 'rsi_avg_departed_diff']
stop_output_cols += [c for c in rsi_check_cols if c in stop_report.columns]
stop_output_cols = [c for c in stop_output_cols if c in stop_report.columns]

stop_report = stop_report[stop_output_cols]
stop_report['ssrs']              = stop_report['ssrs'].round(2)
stop_report['avg_completed_diff'] = stop_report['avg_completed_diff'].round(2)

stop_report.to_csv(PATH + '/stop_level_diagnostic.csv', index=False)
print('stop_level_diagnostic.csv saved —', len(stop_report), 'rows')
stop_report.head()

---
## Summary

### Output files

| File | Granularity | Key fields |
|---|---|---|
| `stop_level_diagnostic.csv` | Per stop | `ssrs`, `avg_completed_diff`, `total_dispatches`, `type_a_count`, stop dimension flags |

### Column changes vs. original plan

| Original | Updated | Reason |
|---|---|---|
| `route_stops.csv` | `route_stops_clean.csv` | Renamed table with more stop-dimension columns |
| `stop_types` from `route_stops.csv` | `stop_types` from `route_stops_clean.csv` | Same field, new source |